# Quarantine Layer - Data Quality Isolation

**Purpose**: Isolate, review, and reprocess records that fail data quality validation to prevent bad data from polluting downstream layers.

**Target Audience**: Data quality analysts, pipeline operators, domain experts

**Layer Position**: Quality gate between Silver and downstream layers (Silver → **Quarantine** → Semantic/Warehouse)

---

## 🚫 Overview

The Quarantine layer provides a **controlled staging area** for problematic records:
* **Isolation** - Failed records don't block pipeline progress
* **Root Cause Analysis** - Human review determines if data is fixable or invalid
* **Reprocessing** - Corrected records flow back into pipeline
* **Discard Tracking** - Permanently rejected records logged for audit
* **Automated Cleanup** - Retention policies prevent unbounded growth

### Architecture Principles

* **Non-Blocking** - Quarantine never stops pipeline execution
* **Versioned** - Track record state changes (PENDING → REPROCESS → DISCARDED)
* **Auditable** - Every decision (reprocess/discard) is logged with reason
* **Recoverable** - Reprocessing capability without re-ingesting from source
* **Time-Bounded** - Retention policies ensure timely resolution or discard

---

## 📁 Quarantine Notebooks

### **Quarantine Lifecycle Management**

```
[Silver Layer] 
     ↓ (DQ Validation)
  ❌ FAILED
     ↓
[quarantine_route_records] ← Route to quarantine
     ↓
[Quarantine Table: PENDING]
     ↓
  Human Review
     ↓
   [✅ Fix] or [❌ Discard]
     ↓
[quarantine_review_apply] ← Apply decision
     ∣
     ├───→ REPROCESS → [quarantine_reprocess_batch]
     │                       ↓
     │                  Back to Silver
     │
     └───→ DISCARDED → [quarantine_retention_cleanup]
                              ↓
                         Archived/Deleted
```

---

#### 1. `quarantine_route_records`
**Purpose**: Route failed DQ validation records to quarantine table  
**Target Table**: `workspace.quarantine.quarantine_jobs`  
**Triggered By**: Silver layer DQ validation (`silver_dq_validate`)  
**Input**: Failed records with DQ violation details  
**Output**: Records inserted into quarantine with `reprocess_status = 'PENDING'`  
**Use Cases**: Isolate bad data, preserve pipeline continuity

**Key Functions**:
* `route_to_quarantine()` - Insert failed records with context
* `enrich_quarantine_metadata()` - Add DQ rule, severity, violation reason
* `notify_review_queue()` - Alert DQ analysts of new quarantine records

**Routing Criteria**:
| DQ Severity | Action |
|-------------|--------|
| ERROR | Route to quarantine |
| WARN | Route to semantic review queue (not quarantine) |
| INFO | Log only, don't quarantine |

---

#### 2. `quarantine_review_apply`
**Purpose**: Apply human review decisions to quarantined records  
**Target Table**: `workspace.quarantine.quarantine_jobs`  
**Input**: CSV/UI with record decisions (`enterprise_job_id`, `decision`, `reason`)  
**Decisions**:
* `REPROCESS` - Fix applied, ready to reprocess
* `DISCARD` - Invalid/duplicate, permanently reject
* `PENDING_INFO` - Needs more context

**Key Functions**:
* `load_review_decisions()` - Parse reviewer input file
* `update_quarantine_status()` - Apply status changes
* `log_review_audit()` - Track who decided what and why
* `trigger_reprocess()` - Automatically call reprocess notebook for REPROCESS records

**Review Decision Format**:
```csv
enterprise_job_id,decision,reason,reviewer
JOB_123,REPROCESS,"Fixed company name typo",analyst@company.com
JOB_456,DISCARD,"Duplicate of JOB_789",analyst@company.com
```

---

#### 3. `quarantine_reprocess_batch`
**Purpose**: Reprocess corrected records back through Silver → Semantic pipeline  
**Input**: Records with `reprocess_status = 'REPROCESS'`  
**Output**: Records flow back into `silver_jobs_current`, status → `REPROCESSED`  
**Workflow**:
1. Extract REPROCESS records from quarantine
2. Re-run Silver standardization
3. Re-run DQ validation (must pass this time)
4. Re-run semantic enrichment
5. Insert into warehouse layer
6. Mark quarantine status as `REPROCESSED`

**Key Functions**:
* `extract_reprocess_batch()` - Get ready-to-reprocess records
* `rerun_silver_pipeline()` - Apply Silver transformations
* `revalidate_dq()` - Verify DQ rules now pass
* `update_reprocess_status()` - Mark as REPROCESSED or REPROCESS_FAILED

**Failure Handling**:
* If reprocessing fails DQ again → Status becomes `REPROCESS_FAILED`
* Alert reviewer: "Fix was insufficient, needs re-review"

---

#### 4. `quarantine_retention_cleanup`
**Purpose**: Automated cleanup of old quarantine records to prevent unbounded growth  
**Target Table**: `workspace.quarantine.quarantine_jobs`  
**Retention Policies**:

| Status | Archive After | Delete After | Notes |
|--------|---------------|--------------|-------|
| REPROCESSED | 30 days | 90 days | Successfully fixed |
| DISCARDED | 7 days | 30 days | Permanently rejected |
| REPROCESS_FAILED | 60 days | 180 days | Needs investigation |
| PENDING | Alert at 60 days | 365 days | Requires review |

**Key Functions**:
* `analyze_retention_status()` - Show age distribution by status
* `archive_records()` - Move to `quarantine_jobs_archive` table
* `delete_archived_records()` - Permanent deletion after retention period
* `alert_stale_pending()` - Flag old PENDING records needing attention

**Safety Features**:
* **Dry Run Mode** - Preview actions before execution
* **Manual Override** - Emergency cleanup by date range
* **Audit Log** - Track all cleanup actions

---

## 🔧 Usage Patterns

### **Pattern 1: Route Failed Records to Quarantine**

Called from Silver DQ validation:

```python
# In silver_dq_validate notebook
failed_df = df.filter(col("dq_status") == "FAILED")

if failed_df.count() > 0:
    # Route to quarantine
    dbutils.notebook.run(
        "../quarantine/quarantine_route_records",
        timeout_seconds=1800,
        arguments={
            "source_table": "silver.silver_jobs_current",
            "failed_records_df": failed_df.toJSON().collect(),
            "dq_rule": "company_name_not_null",
            "severity": "ERROR"
        }
    )
```

---

### **Pattern 2: Apply Review Decisions**

After human review:

```python
# Upload decisions CSV to volume
review_file = "/Volumes/workspace/quarantine/review_decisions_2026-06-05.csv"

# Apply decisions
dbutils.notebook.run(
    "./quarantine_review_apply",
    timeout_seconds=1800,
    arguments={
        "review_file_path": review_file,
        "reviewer_email": "analyst@company.com",
        "auto_trigger_reprocess": "true"  # Automatically reprocess REPROCESS records
    }
)
```

---

### **Pattern 3: Query Quarantine Status**

```sql
-- Current quarantine summary
SELECT 
    reprocess_status,
    COUNT(*) AS record_count,
    MIN(quarantined_at) AS oldest_record,
    MAX(quarantined_at) AS newest_record
FROM workspace.quarantine.quarantine_jobs
GROUP BY reprocess_status;

-- Records pending review (oldest first)
SELECT 
    enterprise_job_id,
    source_name,
    job_title,
    company_name,
    quarantine_reason,
    DATEDIFF(DAY, quarantined_at, CURRENT_TIMESTAMP()) AS days_in_quarantine
FROM workspace.quarantine.quarantine_jobs
WHERE reprocess_status = 'PENDING'
ORDER BY quarantined_at ASC
LIMIT 100;
```

---

### **Pattern 4: Scheduled Cleanup**

Run weekly via job:

```python
# Preview cleanup actions (dry run)
dbutils.notebook.run(
    "./quarantine_retention_cleanup",
    timeout_seconds=3600,
    arguments={
        "dry_run": "true",
        "enable_archive": "true",
        "enable_delete": "true"
    }
)

# After review, run actual cleanup
# (Set dry_run=false in production job)
```

---

## 🔄 Data Flow & Dependencies

### **Quarantine Data Flow**

```
Silver DQ Validation
  ↓ (Failed Records)
quarantine_route_records
  ↓
Quarantine Table (PENDING)
  ↓
Human Review + Decision
  ↓
quarantine_review_apply
  ├───→ REPROCESS → quarantine_reprocess_batch → Back to Silver
  └───→ DISCARDED → quarantine_retention_cleanup → Archive/Delete
```

### **Quarantine Table Schema**

`workspace.quarantine.quarantine_jobs`:

| Column | Type | Description |
|--------|------|-------------|
| `quarantine_id` | STRING | Unique quarantine entry ID |
| `enterprise_job_id` | STRING | FK to silver.silver_jobs_current |
| `source_name` | STRING | Originating API source |
| `job_title` | STRING | Job title (for reference) |
| `company_name` | STRING | Company name (for reference) |
| `quarantine_reason` | STRING | DQ rule that failed |
| `quarantine_severity` | STRING | ERROR, WARN |
| `raw_payload` | STRING | Original JSON payload |
| `reprocess_status` | STRING | PENDING / REPROCESS / REPROCESSED / DISCARDED / REPROCESS_FAILED |
| `reprocess_reason` | STRING | Reviewer's decision reason |
| `reviewed_by` | STRING | Reviewer email |
| `reviewed_at` | TIMESTAMP | Review timestamp |
| `quarantined_at` | TIMESTAMP | Initial quarantine timestamp |
| `reprocessed_at` | TIMESTAMP | Reprocessing timestamp |

### **Status Lifecycle**

```
PENDING → REPROCESS → REPROCESSED (success)
   ↓          ↓
   ↓      REPROCESS_FAILED (DQ failed again)
   ↓
DISCARDED (permanent rejection)
```

---

## ✅ Best Practices

### **Quarantine Routing**

1. **Route ERROR-level Only** - Don't quarantine WARN-level (use semantic review queue)
2. **Include Full Context** - Raw payload, DQ rule, violation details
3. **Preserve Original Timestamps** - Never lose `source_ingestion_timestamp`
4. **Deduplication** - Don't quarantine same record twice for same reason

### **Human Review**

* **Timely Review** - Target: resolve PENDING within 7 days
* **Documented Reasons** - Every decision requires a reason field
* **Batch Decisions** - Use CSV import for bulk review (not one-by-one UI)
* **Domain Expertise** - Route different DQ failures to appropriate reviewers
  * Company name issues → Business analyst
  * Salary range issues → Compensation specialist
  * Location issues → Geography data steward

### **Reprocessing**

* **Validate Fixes** - DQ rules must pass on reprocessing
* **Idempotency** - Reprocessing same record multiple times should be safe
* **Downstream Propagation** - Ensure reprocessed records flow to warehouse/gold
* **Audit Trail** - Track original quarantine → fix → reprocess → production

### **Retention & Cleanup**

* **Automated Cleanup** - Don't rely on manual cleanup
* **Archive Before Delete** - Move to `quarantine_jobs_archive` first
* **Alert Stale PENDING** - Flag records older than 60 days for review
* **Document Cleanup Actions** - Log what was deleted and why

---

## 🚨 Monitoring & Alerts

### **Quarantine Metrics**

| Metric | Target | Alert Threshold |
|--------|--------|----------------|
| PENDING Records | <100 | >500 |
| Avg Time in PENDING | <7 days | >14 days |
| Reprocess Success Rate | >90% | <70% |
| DISCARDED % | <10% of quarantined | >30% |
| PENDING > 60 days | 0 | >10 records |

### **Dashboard Queries**

```sql
-- Quarantine health dashboard
SELECT 
    reprocess_status,
    COUNT(*) AS count,
    ROUND(AVG(DATEDIFF(DAY, quarantined_at, CURRENT_TIMESTAMP())), 1) AS avg_days_in_quarantine,
    MAX(DATEDIFF(DAY, quarantined_at, CURRENT_TIMESTAMP())) AS max_days_in_quarantine
FROM workspace.quarantine.quarantine_jobs
GROUP BY reprocess_status;

-- Top quarantine reasons
SELECT 
    quarantine_reason,
    COUNT(*) AS occurrences,
    COUNT(CASE WHEN reprocess_status = 'REPROCESSED' THEN 1 END) AS reprocessed,
    COUNT(CASE WHEN reprocess_status = 'DISCARDED' THEN 1 END) AS discarded
FROM workspace.quarantine.quarantine_jobs
WHERE quarantined_at >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY quarantine_reason
ORDER BY occurrences DESC;
```

### **Alert Triggers**

* **High Quarantine Volume** - >500 PENDING records (source data quality degraded)
* **Stale PENDING Records** - >10 records older than 60 days (review backlog)
* **Low Reprocess Success** - <70% success rate (fixes inadequate)
* **High Discard Rate** - >30% discarded (upstream data quality issue)

---

## 🔧 Troubleshooting

### **Issue: Reprocessing fails with same DQ error**

**Symptoms**: Records stuck in `REPROCESS_FAILED` status  
**Causes**:
* Fix was insufficient
* DQ rule threshold changed
* Reviewer misunderstood failure reason

**Solution**:
```sql
-- Find reprocess failures
SELECT 
    enterprise_job_id,
    quarantine_reason,
    reprocess_reason,
    reprocessed_at
FROM workspace.quarantine.quarantine_jobs
WHERE reprocess_status = 'REPROCESS_FAILED'
ORDER BY reprocessed_at DESC;

-- Review original DQ violation and fix
-- Update reprocess_status back to PENDING for re-review
UPDATE workspace.quarantine.quarantine_jobs
SET reprocess_status = 'PENDING',
    reprocess_reason = 'Fix insufficient, needs re-review'
WHERE quarantine_id = 'Q12345';
```

---

### **Issue: PENDING records piling up**

**Symptoms**: >500 PENDING records, review backlog growing  
**Causes**:
* Systematic data quality issue at source
* Review process too slow
* No assigned reviewer

**Solution**:
1. **Investigate Root Cause**:
```sql
-- Top quarantine reasons
SELECT quarantine_reason, COUNT(*)
FROM workspace.quarantine.quarantine_jobs
WHERE reprocess_status = 'PENDING'
GROUP BY quarantine_reason
ORDER BY COUNT(*) DESC;
```

2. **Bulk Discard** (if data is invalid):
```sql
-- Mark all records failing specific rule as DISCARDED
UPDATE workspace.quarantine.quarantine_jobs
SET reprocess_status = 'DISCARDED',
    reprocess_reason = 'Bulk discard: source data format changed',
    reviewed_by = 'system@company.com',
    reviewed_at = CURRENT_TIMESTAMP()
WHERE reprocess_status = 'PENDING'
  AND quarantine_reason = 'invalid_salary_format';
```

3. **Fix at Source** - Collaborate with API provider to improve data quality

---

### **Issue: Cleanup deleted records still needed**

**Symptoms**: Accidentally deleted records that should have been reprocessed  
**Causes**: Cleanup ran without proper review

**Solution**:
```sql
-- Check archive table
SELECT * FROM workspace.quarantine.quarantine_jobs_archive
WHERE enterprise_job_id = 'JOB_123';

-- Restore from archive
INSERT INTO workspace.quarantine.quarantine_jobs
SELECT * FROM workspace.quarantine.quarantine_jobs_archive
WHERE enterprise_job_id = 'JOB_123';

-- Update status to REPROCESS
UPDATE workspace.quarantine.quarantine_jobs
SET reprocess_status = 'REPROCESS',
    reprocess_reason = 'Restored from archive for reprocessing'
WHERE enterprise_job_id = 'JOB_123';
```

---

## 📚 Related Documentation

* **Silver DQ Validation** - See `notebooks/silver/silver_dq_validate` for DQ rules that trigger quarantine
* **Semantic Review Queue** - See `notebooks/semantic/semantic_review_resolver` for WARN-level issues
* **Audit Logging** - See `notebooks/audit/README_AUDIT` for quarantine event tracking
* **Data Quality Framework** - See `docs/data_quality_framework.md` for DQ rule definitions

---

## 🏁 Quick Start

### **1. Check Current Quarantine Status**
```sql
SELECT reprocess_status, COUNT(*)
FROM workspace.quarantine.quarantine_jobs
GROUP BY reprocess_status;
```

### **2. Review Pending Records**
```sql
SELECT * FROM workspace.quarantine.quarantine_jobs
WHERE reprocess_status = 'PENDING'
ORDER BY quarantined_at ASC
LIMIT 10;
```

### **3. Apply Review Decisions**
```python
dbutils.notebook.run(
    "./quarantine_review_apply",
    1800,
    {"review_file_path": "/Volumes/workspace/quarantine/reviews.csv"}
)
```

### **4. Run Reprocessing**
```python
dbutils.notebook.run("./quarantine_reprocess_batch", 3600)
```

---

**Last Updated**: 2026-06-05  
**Maintained By**: Data Quality Team  
**Questions?**: Contact #data-quality on Slack